In [11]:
import h5py

# File paths for input and output
input_h5_path = "/Users/didemdost/Desktop/reduced_embeddings_file_ProstT5.h5"
reference_h5_path = "/Users/didemdost/Desktop/per_protein_embeddings_c0.3.h5"
output_h5_path = "/Users/didemdost/Desktop/restructured_embeddings_file_ProstT5.h5"

# Open the reference file to understand the structure
with h5py.File(reference_h5_path, "r") as ref_h5:
    print("Reference HDF5 File Structure:")
    def traverse_and_print(group, indent=0):
        for key in group.keys():
            obj = group[key]
            if isinstance(obj, h5py.Group):
                print("  " * indent + f"Group: {key}")
                traverse_and_print(obj, indent + 1)
            elif isinstance(obj, h5py.Dataset):
                print("  " * indent + f"Dataset: {key}")
                print("  " * (indent + 1) + f"Shape: {obj.shape}")
                print("  " * (indent + 1) + f"Data type: {obj.dtype}")

    traverse_and_print(ref_h5)

# Now process the input file and restructure it based on the reference structure
with h5py.File(input_h5_path, "r") as input_h5, \
     h5py.File(output_h5_path, "w") as output_h5:

    # Restructure based on the reference
    for dataset_name in input_h5.keys():
        dataset = input_h5[dataset_name]

        # Use `original_id` attribute if available, else skip
        original_id = dataset.attrs.get("original_id", None)
        if original_id:
            new_dataset_name = original_id.decode("utf-8") if isinstance(original_id, bytes) else original_id
            output_h5.create_dataset(new_dataset_name, data=dataset[:], dtype=dataset.dtype)

print(f"\nConverted HDF5 file saved to: {output_h5_path}")

# Inspect the structure of the new file
with h5py.File(output_h5_path, "r") as output_h5:
    print("\nNewly Created HDF5 File Structure:")
    traverse_and_print(output_h5)


Reference HDF5 File Structure:
Dataset: A0A0R4J737
  Shape: (1024,)
  Data type: float32
Dataset: A0A0R4J7V3
  Shape: (1024,)
  Data type: float32
Dataset: A0A1V4PWS3
  Shape: (1024,)
  Data type: float32
Dataset: A0A2X3U5V2
  Shape: (1024,)
  Data type: float32
Dataset: A0A3T1A1W4
  Shape: (1024,)
  Data type: float32
Dataset: A0A656GG29
  Shape: (1024,)
  Data type: float32
Dataset: A0A656GNA3
  Shape: (1024,)
  Data type: float32
Dataset: A0A656GRZ5
  Shape: (1024,)
  Data type: float32
Dataset: A0A656JK29
  Shape: (1024,)
  Data type: float32
Dataset: A0A656JKS4
  Shape: (1024,)
  Data type: float32
Dataset: A0A656JMP9
  Shape: (1024,)
  Data type: float32
Dataset: A0A656JXF7
  Shape: (1024,)
  Data type: float32
Dataset: A0A656JXQ0
  Shape: (1024,)
  Data type: float32
Dataset: A0A6B0D2Z3
  Shape: (1024,)
  Data type: float32
Dataset: A0A6G7B150
  Shape: (1024,)
  Data type: float32
Dataset: A0FDW6
  Shape: (1024,)
  Data type: float32
Dataset: A0FKE4
  Shape: (1024,)
  Data type:

In [13]:
import h5py

# File path for the converted HDF5 file
output_h5_path = "/Users/didemdost/Desktop/restructured_embeddings_file_ProstT5.h5"

# Modify dataset IDs in the output file
with h5py.File(output_h5_path, "r+") as h5_file:
    datasets_to_modify = list(h5_file.keys())  # Collect dataset names
    for dataset_name in datasets_to_modify:
        new_name = dataset_name.replace("_", ".", 1)  # Replace the second `_` with `.`
        if new_name != dataset_name:
            h5_file.move(dataset_name, new_name)  # Rename dataset
            print(f"Renamed {dataset_name} to {new_name}")

print("\nAll dataset IDs updated in the HDF5 file.")


Renamed AAC28881_1 to AAC28881.1
Renamed AAC69806_1 to AAC69806.1
Renamed AAD16811_1 to AAD16811.1
Renamed AAF71481_2 to AAF71481.2
Renamed AAG01467_2 to AAG01467.2
Renamed AAK69208_1 to AAK69208.1
Renamed AAL20326_1 to AAL20326.1
Renamed AAL52397_1 to AAL52397.1
Renamed AAN29664_1 to AAN29664.1
Renamed AAN37523_1 to AAN37523.1
Renamed AAO18426_1 to AAO18426.1
Renamed AAO68326_1 to AAO68326.1
Renamed AAP31245_1 to AAP31245.1
Renamed AAP34334_1 to AAP34334.1
Renamed AAQ60244_1 to AAQ60244.1
Renamed AAQ75736_1 to AAQ75736.1
Renamed AAR21118_1 to AAR21118.1
Renamed AAR26341_1 to AAR26341.1
Renamed AAR84051_1 to AAR84051.1
Renamed AAS20351_1 to AAS20351.1
Renamed AAS47019_1 to AAS47019.1
Renamed AAS47020_1 to AAS47020.1
Renamed AAS48168_1 to AAS48168.1
Renamed AAS48169_1 to AAS48169.1
Renamed AAS58577_1 to AAS58577.1
Renamed AAS66851_1 to AAS66851.1
Renamed AAS66853_1 to AAS66853.1
Renamed AAS91821_1 to AAS91821.1
Renamed AAT35179_1 to AAT35179.1
Renamed AAU95471_1 to AAU95471.1
Renamed AA

In [17]:
import pandas as pd

# File paths
csv_path = "/Users/didemdost/Desktop/output_original/ToxinTypes_labelTarget_3.csv"  # Update with your actual CSV file path
output_merged_path = "/Users/didemdost/Desktop/merged_embeddings_with_labels.csv"

# Read the CSV file with labels
labels_df = pd.read_csv(csv_path, delimiter=";")
print(labels_df)
print("Labels DataFrame loaded successfully.")

# Open the HDF5 file and merge with the labels
merged_data = []
with h5py.File(output_h5_path, "r") as h5_file:
    for dataset_name in h5_file.keys():
        if dataset_name in labels_df['ID'].values:
            embedding = h5_file[dataset_name][:]
            label_row = labels_df[labels_df['ID'] == dataset_name].iloc[0]
            merged_data.append({'ID': dataset_name, 'Embedding': embedding.tolist(), **label_row.to_dict()})

# Convert merged data to a DataFrame
merged_df = pd.DataFrame(merged_data)

# Save the merged DataFrame to a CSV file
merged_df.to_csv(output_merged_path, index=False)
print(f"Merged data saved to {output_merged_path}")


          ID                   species                 type          target
0     G0CIT6    Xanthomonas campestris              Unknown  Non-vertebrate
1     B6A882      Yersinia entomophaga              Unknown  Non-vertebrate
2     B6A881      Yersinia entomophaga              Unknown  Non-vertebrate
3     P22522          Escherichia coli  TypeII_PFT_bacteria  Non-vertebrate
4     Q841V4          Escherichia coli  TypeII_PFT_bacteria  Non-vertebrate
...      ...                       ...                  ...             ...
2487  D0EM77      Tannerella forsythia              Type_IV      Vertebrate
2488  P23694       Serratia marcescens              Type_IV      Vertebrate
2489  Q97T80  Streptococcus pneumoniae              Type_IV      Vertebrate
2490  B1LKV6          Escherichia coli              Type_IV      Vertebrate
2491  Q11137         Proteus mirabilis              Type_IV      Vertebrate

[2492 rows x 4 columns]
Labels DataFrame loaded successfully.
Merged data saved to /Use

In [18]:
import h5py
import pandas as pd
import numpy as np

# File paths
csv_path = "/Users/didemdost/Desktop/merged_embeddings_with_labels.csv"  # Path to the CSV file
h5_output_path = "/Users/didemdost/Desktop/merged_embeddings_with_labels_formatted.h5"  # Path for the output HDF5 file

# Load the CSV file
merged_df = pd.read_csv(csv_path)
print("CSV file loaded successfully.")

# Open an HDF5 file for writing
with h5py.File(h5_output_path, "w") as h5_file:
    for _, row in merged_df.iterrows():
        # Get the ID and embedding
        dataset_name = row["ID"]  # Dataset name from the "ID" column
        embedding = np.array(row["Embedding"].strip("[]").split(", "), dtype=np.float32)  # Convert string to array

        # Create a dataset with the ID as the name
        h5_file.create_dataset(dataset_name, data=embedding, dtype="float32")
        print(f"Created dataset: {dataset_name}, Shape: {embedding.shape}")

print(f"\nFormatted HDF5 file saved at: {h5_output_path}")


CSV file loaded successfully.
Created dataset: A0A068QWU7, Shape: (1024,)
Created dataset: A0A068QYK9, Shape: (1024,)
Created dataset: A0A0A8J2Q4, Shape: (1024,)
Created dataset: A0A0C5XL88, Shape: (1024,)
Created dataset: A0A0C6PDA9, Shape: (1024,)
Created dataset: A0A0F7RBI7, Shape: (1024,)
Created dataset: A0A0H3AIG7, Shape: (1024,)
Created dataset: A0A0H3CC30, Shape: (1024,)
Created dataset: A0A0H3KDT7, Shape: (1024,)
Created dataset: A0A0H3KHN8, Shape: (1024,)
Created dataset: A0A0H3MD02, Shape: (1024,)
Created dataset: A0A0H3MGR4, Shape: (1024,)
Created dataset: A0A0K5ZZ47, Shape: (1024,)
Created dataset: A0A0N2IV24, Shape: (1024,)
Created dataset: A0A0R4FQ94, Shape: (1024,)
Created dataset: A0A0R4FRR9, Shape: (1024,)
Created dataset: A0A0R4J6H0, Shape: (1024,)
Created dataset: A0A0R4J6P7, Shape: (1024,)
Created dataset: A0A0R4J737, Shape: (1024,)
Created dataset: A0A0R4J7V3, Shape: (1024,)
Created dataset: A0A0R4J835, Shape: (1024,)
Created dataset: A0A1V4PWS3, Shape: (1024,)
Cr

In [19]:
import os
import pandas as pd

# Paths
folder_path = "/Users/didemdost/Desktop/folds/folds/mergedExotoxins_folds"  # Path to the folder
csv_path = "/Users/didemdost/Desktop/merged_embeddings_with_labels.csv"  # Path to the CSV file
output_csv_path = "/Users/didemdost/Desktop/filtered_embeddings_with_labels.csv"  # Output CSV file path

# Step 1: Extract filenames (excluding .pdb)
processed_filenames = [
    os.path.splitext(file)[0]  # Remove the .pdb extension
    for file in os.listdir(folder_path)
    if file.endswith(".pdb")  # Only consider .pdb files
]
print(f"Extracted filenames (without .pdb): {processed_filenames}")

# Step 2: Load the CSV file
csv_data = pd.read_csv(csv_path)
print(f"Original CSV file loaded with {len(csv_data)} rows.")

# Step 3: Filter rows in the CSV based on IDs matching the filenames
filtered_data = csv_data[csv_data["ID"].isin(processed_filenames)]
print(f"Filtered CSV file contains {len(filtered_data)} rows.")

# Step 4: Save the filtered data to a new CSV file
filtered_data.to_csv(output_csv_path, index=False)
print(f"Filtered CSV file saved to: {output_csv_path}")

Extracted filenames (without .pdb): ['ABS76498.1', 'YP.096462.1', 'U6B8P7', 'Q4G4C8', 'Q889A9', 'NP.820668.1', 'NP.819986.1', 'Q97T80', 'P23874', 'P64524', 'Q5ZYJ6', 'NP.820359.1', 'Q8FDW4', 'I6VBE8', 'Q9Z3F4', 'F0V195', 'Q93KQ5', 'S3MY93', 'C0SPQ6', 'P0CZ04', 'YP.095729.1', 'Q8RP04', 'Q881L7', 'Q5ZSE2', 'A7JZC6', 'P0C1V1', 'ACJ21157.1', 'CAA12187.1', 'NP.820802.1', 'CAM11323.1', 'Q5ZSK8', 'Q5ZT91', 'Q1I8U1', 'Q50EL2', 'P09332', 'F3FD65', 'Q9Z8X8', 'B4SX34', 'YP.003366672.1', 'BAH47296.1', 'YP.096814.1', 'A0FKE4', 'A9CGH1', 'Q5ZSQ6', 'YP.094449.1', 'Q87V79', 'Q79UN8', 'Q5ZYJ7', 'AAW70656.1', 'Q5ZWD3', 'P31518', 'Q5ZWI4', 'Q5ZU32', 'YP.094100.1', 'P0C0Q4', 'NP.820476.2', 'P04979', 'C8BNW8', 'Q8XC86', 'Q84CS9', 'Q5ZR83', 'Q45716', 'Q02307', 'Q56026', 'P40136', 'Q5ZRN6', 'YP.415003.1', 'P11437', 'Q5ZUE7', 'D4HWL8', 'ABS78513.1', 'Q5ZYS7', 'Q5ZU30', 'O84951', 'P54355', 'E7G765', 'P55711', 'YP.096188.1', 'Q5ZWD1', 'ACJ17987.1', 'Q5ZVN2', 'A1JQ83', 'P55117', 'O87653', 'Q87WF8', 'P64453', 'F3

In [22]:
# Check the row number of the CSV file
csv_path = "/Users/didemdost/Desktop/filtered_embeddings_with_labels.csv"

# Load the CSV file
try:
    csv_data = pd.read_csv(csv_path)
    row_count = len(csv_data)
    print(f"The CSV file contains {row_count} rows.")
except FileNotFoundError:
    print(f"The file at {csv_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


The CSV file contains 872 rows.


In [23]:
import h5py
import pandas as pd
import numpy as np

# File paths
csv_path = "/Users/didemdost/Desktop/filtered_embeddings_with_labels.csv"  
h5_output_path = "/Users/didemdost/Desktop/folds_with_labels.h5" 

# Load the CSV file
merged_df = pd.read_csv(csv_path)
print("CSV file loaded successfully.")

# Open an HDF5 file for writing
with h5py.File(h5_output_path, "w") as h5_file:
    for _, row in merged_df.iterrows():
        # Get the ID and embedding
        dataset_name = row["ID"]  # Dataset name from the "ID" column
        embedding = np.array(row["Embedding"].strip("[]").split(", "), dtype=np.float32)  # Convert string to array

        # Create a dataset with the ID as the name
        h5_file.create_dataset(dataset_name, data=embedding, dtype="float32")
        print(f"Created dataset: {dataset_name}, Shape: {embedding.shape}")

print(f"\nFormatted HDF5 file saved at: {h5_output_path}")


CSV file loaded successfully.


KeyError: 'ID'